In [0]:
import sys
from pathlib import Path

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = Path("/Workspace") / Path(notebook_path).parent
except Exception:
    repo_root = Path.cwd()

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from config import (
    BRONZE_CONTENTS_PATH,
    BRONZE_LOGS_PATH,
    BRONZE_OTT_CONTENTS_TABLE,
    BRONZE_OTT_LOGS_TABLE,
    BRONZE_OTT_USERS_TABLE,
    BRONZE_USERS_PATH,
)

# ── 1. 원본 읽기 ──────────────────────────────────────────

file_path_contents = BRONZE_CONTENTS_PATH
file_path_logs = BRONZE_LOGS_PATH
file_path_users = BRONZE_USERS_PATH


# ── 2. 스키마 지정 ──────────────────────────────────────────

schema_contents = StructType([
    StructField("content_id", StringType(), True),
    StructField("content_type", StringType(), True),
    StructField("genre", StringType(), True),
    StructField("total_duration_sec", IntegerType(), True)
])

schema_logs = StructType([
    StructField("log_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("content_id", StringType(), True),
    StructField("view_timestamp", TimestampType(), True),
    StructField("watched_seconds", IntegerType(), True),
    StructField("device", StringType(), True)
])

schema_users = StructType([
    StructField("user_id", StringType(), True),
    StructField("plan_type", StringType(), True),
    StructField("country", StringType(), True),
    StructField("signup_date", DateType(), True)
])

# ── 3. 데이터프레임 생성 ──────────────────────────────────────────

df_file_path_contents = spark.read.csv(
    file_path_contents, 
    header=True, 
    schema=schema_contents
)

df_file_path_logs = spark.read.csv(
    file_path_logs, 
    header=True, 
    schema=schema_logs
)

df_file_path_users = spark.read.csv(
    file_path_users, 
    header=True, 
    schema=schema_users
)

# ── 3.1 데이터프레임 확인 ──────────────────────────────────────────

# display(df_file_path_contents.limit 3)
# display(df_file_path_logs.limit 3)
# display(df_file_path_users.limit 3)

In [0]:
# 4. 테이블로 저장 ──────────────────────────────────────────

(
    df_file_path_contents
    .write
    .mode("overwrite")
    .saveAsTable(BRONZE_OTT_CONTENTS_TABLE)
)

(
    df_file_path_logs
    .write
    .mode("overwrite")
    .saveAsTable(BRONZE_OTT_LOGS_TABLE)
)

(
    df_file_path_users
    .write
    .mode("overwrite")
    .saveAsTable(BRONZE_OTT_USERS_TABLE)
)

In [0]:
print(f"{BRONZE_OTT_CONTENTS_TABLE}: {spark.table(BRONZE_OTT_CONTENTS_TABLE).count()} rows")
print(f"{BRONZE_OTT_LOGS_TABLE}: {spark.table(BRONZE_OTT_LOGS_TABLE).count()} rows")
print(f"{BRONZE_OTT_USERS_TABLE}: {spark.table(BRONZE_OTT_USERS_TABLE).count()} rows")


count(*)
100500



- inferSchema 기능은 데이터 타입을 알아서 정해줌 

- 실무에선 inferSchema보단 스키마를 직접 지정하는 게 좋음 왜냐하면 오류가 날 수 있기 때문

- 추가로 발표하면서 다른 사람들의 질문을 받으니 명시해놓는 게 협업할 때 좋을 거라 생각이 들었음 